# Text generation with transformers from Hugging Face 🤗

## Setup
- activate environment: conda activate hugvenv312
- install requirements: pip install -r requirements.txt

In [3]:
# Check CPU, GPU, and ROCm installation
import platform  # For system/platform info
import torch     # For PyTorch and GPU checks
import subprocess  # For running shell commands

# Print platform and Python version
print(f'Platform: {platform.platform()}')
print(f'Python version: {platform.python_version()}')

# Print PyTorch version and check CUDA (NVIDIA GPU) availability
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device count: {torch.cuda.device_count()}')
    print(f'CUDA device name: {torch.cuda.get_device_name(0)}')
else:
    print('No CUDA GPU detected.')

# Try to print ROCm version (for AMD GPU support)
try:
    result = subprocess.run(['rocm-smi', '--showversion'], capture_output=True, text=True)
    print('ROCm SMI output:')
    print(result.stdout)
except Exception as e:
    print('ROCm SMI not found or not installed.')

Platform: Windows-11-10.0.26200-SP0
Python version: 3.12.13
PyTorch version: 2.9.1+rocm7.2.1
CUDA available: True
CUDA device count: 1
CUDA device name: AMD Radeon RX 7900 XT
ROCm SMI not found or not installed.


## Version 1 - Using the default pipeline as a high-level helper from HugginFace 🤗

In [4]:
from transformers import pipeline  # Import the pipeline helper

# Create a text-generation pipeline using the GPT-2 model from Hugging Face Hub
pipe = pipeline('text-generation', model='openai-community/gpt2')

# Define a prompt for the model to complete
prompt = 'What is machine learning?'
# Generate text using the pipeline
output = pipe(prompt)
# 'output' is a list of dicts with 'generated_text' key

c:\Users\lovep\miniconda3\envs\hugenv312\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [5]:
# Display the generated text from the pipeline output
output[0]['generated_text']

'What is machine learning? How important is it to you to understand machine learning and how is it helping people to be creative? How far away will you go?\n\nDo you need any background in machine learning to help you? If so, please'

## Version 2 - Using the model and tokenizer directly for more control

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM  # Direct model/tokenizer usage

# Load the tokenizer and model directly for more control
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2')
model = AutoModelForCausalLM.from_pretrained('openai-community/gpt2')

tokenizer # Display the tokenizer object to confirm it loaded correctly

GPT2TokenizerFast(name_or_path='openai-community/gpt2', vocab_size=50257, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}

In [7]:
# Encode a sentence to input IDs (token IDs) for the model
sentence = 'What is machine learning?'
input_ids = tokenizer(sentence, return_tensors='pt')["input_ids"]
input_ids  # Tensor of token IDs

tensor([[2061,  318, 4572, 4673,   30]])

In [8]:
# Decode the input IDs back to text (should match the original sentence)
tokenizer.decode(input_ids[0])

'What is machine learning?'

### Tokenization breakdown for 'unbelievable'
This cell prints each token ID and its corresponding string for the word 'unbelievable'.
You can see how the tokenizer splits the word into subword tokens.


In [9]:
# Try encoding a different word to see its token IDs
sentence = 'unbelievable'
input_ids = tokenizer(sentence, return_tensors='pt').input_ids
input_ids  # Tensor of token IDs

tensor([[  403,  6667, 11203,   540]])

In [10]:
# Print each token ID and its corresponding decoded string for the word 'unbelievable'
for token_id in input_ids[0]:
    print(f'Token ID {token_id} is in the input {tokenizer.decode(token_id)}')

Token ID 403 is in the input un
Token ID 6667 is in the input bel
Token ID 11203 is in the input iev
Token ID 540 is in the input able


### Tokenization breakdown for 'homoscendasticity'
This cell demonstrates how the tokenizer handles a rare or complex word.
It prints each token ID and its decoded string, then decodes the full sequence back to text.
Notice how the tokenizer may split the word into multiple subword tokens if it is not in the vocabulary.

In [11]:
# Try a rare or complex word to see how the tokenizer splits it
word = 'homoscendasticity'
my_ids = tokenizer(word, return_tensors='pt').input_ids
my_ids  # Tensor of token IDs

# Print each token ID and its decoded string
for token_id in my_ids[0]:
    print(f'Token ID: {token_id} for input --> {tokenizer.decode(token_id)}')

# Decode the full sequence of token IDs back to text
tokenizer.decode(my_ids.squeeze())

Token ID: 26452 for input --> hom
Token ID: 418 for input --> os
Token ID: 15695 for input --> cend
Token ID: 3477 for input --> astic
Token ID: 414 for input --> ity


'homoscendasticity'

Generating text with the model:
This cell generates text based on a given prompt using the GPT-2 model.

In [12]:
gpt2 = AutoModelForCausalLM.from_pretrained('openai-community/gpt2')  # Load the GPT-2 model
gpt2  # Display the model architecture to confirm it loaded correctly

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [13]:
sentence = 'I like machine learning to be able to predict the'
# Encode the sentence to input IDs and pass it through the model to get output logits
sentence_ids = tokenizer(sentence, return_tensors='pt').input_ids

gpt2(sentence_ids)  # Pass the input IDs through the model to get output logits

CausalLMOutputWithCrossAttentions(loss=None, logits=tensor([[[ -39.3084,  -39.0101,  -41.8375,  ...,  -46.9338,  -44.9074,
           -39.5149],
         [ -73.5276,  -72.8394,  -79.3030,  ...,  -80.4989,  -81.8446,
           -76.4412],
         [ -86.4650,  -84.4242,  -92.1375,  ...,  -91.6711,  -93.1266,
           -89.0066],
         ...,
         [-149.8200, -148.6029, -155.0193,  ..., -156.6912, -158.7239,
          -150.9127],
         [-125.7309, -124.6376, -134.1440,  ..., -130.8506, -134.5271,
          -128.6164],
         [-132.9183, -131.4491, -139.0226,  ..., -133.7860, -136.5529,
          -132.2061]]], grad_fn=<UnsafeViewBackward0>), past_key_values=((tensor([[[[-1.5577e+00,  2.0585e+00,  1.3060e+00,  ..., -1.3825e+00,
           -6.3336e-01,  1.2624e+00],
          [-2.5532e+00,  2.6298e+00,  1.7484e+00,  ..., -8.9923e-01,
           -1.2752e+00,  1.8641e+00],
          [-1.4545e+00,  3.5979e+00,  1.4520e+00,  ..., -1.0537e+00,
           -1.9598e+00,  1.4160e+00],
   

In [14]:
# The logits are the raw output scores from the model before applying any activation function (like softmax).
output = gpt2(sentence_ids).logits [0,-1] # Access the logits from the model output
tokenizer.decode(output.argmax())  # Decode the token ID with the highest logit/prediction score to get the predicted next word

' future'

In [15]:
sentence = 'I like machine learning to enchance our understanding of the'
sentence_ids = tokenizer(sentence, return_tensors='pt').input_ids

gpt2(sentence_ids)

output = gpt2(sentence_ids).logits [0,-1]
tokenizer.decode(output.argmax())

' world'

In [20]:
sentence = 'I learn machine learning to enchance my'
token_ids = tokenizer(sentence, return_tensors='pt').input_ids

output = gpt2(token_ids).logits [0,-1]
finql_logits = torch.topk(output, k=10)  # Get the top 10 predicted token IDs and their logits

# Display the top k logits and their corresponding token IDs
for index in finql_logits.indices:
    print(f'Token ID: {index} --> {tokenizer.decode(index)}')

Token ID: 898 -->  own
Token ID: 1204 -->  life
Token ID: 3632 -->  brain
Token ID: 835 -->  way
Token ID: 2003 -->  future
Token ID: 670 -->  work
Token ID: 3451 -->  career
Token ID: 1693 -->  job
Token ID: 1306 -->  next
Token ID: 2000 -->  mind
